# HW1 — Linear regression from scratch

Solution for Assignment 1 in `01_linear_regression__concepts.qmd`.

**Rules.** No scikit-learn. Only numpy / pandas / plotly.

**Outline.**
1. Generate + visualize the data
2. Gradient descent for the one-parameter model $\hat y = \theta \cdot x$
3. Adding an intercept: the design-matrix trick
4. Tune the learning rate
5. A quick taste of SGD
6. Normal equation (closed form) + geometric sanity check
7. Use the model — predict on new $x$
8. Polynomial regression (and what breaks numerically)
9. Bonus: residual histogram, risk-surface trajectory, vectorization sanity-check

In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Armenian flag colors for multi-series plots
ARM_RED = "#D90012"
ARM_BLUE = "#0033A0"
ARM_ORANGE = "#F2A800"

SEED = 509
np.random.seed(SEED)

pio.renderers.default = "notebook_connected" 

## 1. Generate the data

Instead of just reading the CSV, here's the code that produced it. Knowing how the data was generated is useful — we know the true intercept and slope, so we can check whether our fitted model recovers them.

True model: $y = 509 + 509 \cdot x + \text{noise}$, with $x \sim \text{Uniform}(0, 2)$ and Gaussian noise of standard deviation 20.

In [2]:
np.random.seed(SEED)
n_samples = 100
x_arr = 2 * np.random.rand(n_samples)            # x in [0, 2]
noise = np.random.randn(n_samples) * 20.0
y_arr = 509 + 509 * x_arr + noise                # true: intercept=509, slope=509

data = pd.DataFrame({"x": x_arr, "y": y_arr})
print(data.shape)
data.head()

(100, 2)


,x,y
0,1.511746,1289.286331
1,0.501757,777.512816
2,1.411676,1234.708865
3,0.755341,881.945225
4,1.445801,1250.230481


In [3]:
data.describe()

,x,y
count,100.000000,100.000000
mean,1.018819,1027.860618
std,0.591890,301.271265
min,0.026296,499.894639
25%,0.485648,768.249891
50%,1.045391,1045.656147
75%,1.521832,1294.122375
max,1.964979,1513.995602


In [4]:
fig = px.scatter(data, x="x", y="y",
                 title=f"data_lin_reg.csv ({len(data)} points) — true model: y = 509 + 509x + noise")
fig.update_traces(marker=dict(color=ARM_BLUE, size=7, opacity=0.7))
fig.update_layout(width=800, height=450)
fig

Notice the data cloud sits well above $y = 0$ — at $x = 0$ the $y$ values are around 509, not 0. Any line forced to pass through the origin will badly miss the trend. We'll see that explicitly in the next section.

## 2. Gradient descent for the simplest model

Let's start with the simplest possible model: a line that passes through the origin,

$$\hat y = \theta \cdot x$$

just one parameter (the slope). The empirical risk we want to minimize is mean squared error:

$$R(\theta) = \frac{1}{n} \sum_{i=1}^n (\theta x_i - y_i)^2$$

Differentiate with respect to $\theta$:

$$\frac{dR}{d\theta}
   = \frac{1}{n} \sum_{i=1}^n 2(\theta x_i - y_i) \cdot x_i
   = \frac{2}{n} \sum_{i=1}^n (\theta x_i - y_i)\, x_i$$

Gradient descent update with learning rate $\alpha$:

$$\theta^{(t+1)} = \theta^{(t)} - \alpha \, \frac{dR}{d\theta}\big(\theta^{(t)}\big)$$

In [5]:
def risk_1p(x, y, theta):
    """Empirical risk for the one-parameter model y_hat = theta * x."""
    return float(np.mean((theta * x - y) ** 2))


def gradient_1p(x, y, theta):
    """Gradient of the one-parameter risk (a single number)."""
    return (2 / len(y)) * np.sum((theta * x - y) * x)


def gd_1p(x, y, alpha=0.05, n_iter=2000):
    """Run GD for the one-parameter model. Returns final theta + history."""
    theta = 0.0
    history = {"iter": [], "risk": [], "theta": []}
    for t in range(n_iter):
        theta = theta - alpha * gradient_1p(x, y, theta)
        history["iter"].append(t)
        history["risk"].append(risk_1p(x, y, theta))
        history["theta"].append(theta)
    return theta, history


theta_1p, hist_1p = gd_1p(x_arr, y_arr, alpha=0.05, n_iter=2000)
print(f"Final theta after 2000 iterations: {theta_1p:.4f}")
print(f"Final risk: {hist_1p['risk'][-1]:.4f}")

Final theta after 2000 iterations: 883.3969
Final risk: 65651.9143


Instead of printing the risk every few hundred iterations, let's plot how it dropped over time:

In [6]:
fig = px.line(x=hist_1p["iter"], y=hist_1p["risk"], log_y=True,
              labels={"x": "iteration", "y": "empirical risk (log scale)"},
              title="GD convergence — one-parameter model, alpha=0.05")
fig.update_traces(line=dict(color=ARM_BLUE, width=2))
fig.update_layout(width=800, height=420)
fig

And here's the fitted line on top of the data:

In [7]:
# Extend the grid to start at x=0 so we can see the line hitting the origin
x_grid = np.linspace(0, x_arr.max(), 200)
y_hat_1p = theta_1p * x_grid

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_arr, y=y_arr, mode="markers",
                         marker=dict(color=ARM_BLUE, size=7, opacity=0.55),
                         name="data"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_1p, mode="lines",
                         line=dict(color=ARM_RED, width=3),
                         name=f"one-parameter fit: y = {theta_1p:.2f} * x"))
fig.update_layout(title="One-parameter linear regression (line forced through the origin)",
                  xaxis_title="x", yaxis_title="y",
                  width=800, height=450)
fig

The fitted line passes through $(0, 0)$ because our model is $y = \theta x$. The data sits hundreds of units above that — so the best slope GD can find ($\theta \approx 880$) is a compromise (too steep, to compensate for not being able to shift up). Time to add an intercept.

## 3. Adding the intercept — the design-matrix trick

We want the more general model

$$\hat y = \theta_0 + \theta_1 \cdot x$$

One option: carry around two separate variables $\theta_0, \theta_1$, write a separate update rule for each, etc. Doable but tedious — and it gets much worse when we have many features.

The clean trick: **prepend a column of 1s to our features**. If our original feature vector is $x = (x_1, x_2, \ldots, x_n)$, build the matrix

$$X = \begin{pmatrix} 1 & x_1 \\ 1 & x_2 \\ \vdots & \vdots \\ 1 & x_n \end{pmatrix}$$

Then with $\theta = (\theta_0, \theta_1)^\top$, predictions are just $X \theta$:

$$X \theta = \begin{pmatrix} \theta_0 + \theta_1 x_1 \\ \theta_0 + \theta_1 x_2 \\ \vdots \\ \theta_0 + \theta_1 x_n \end{pmatrix}$$

The intercept is absorbed as just another coefficient. One matrix multiplication handles everything; no special-case code.

In [8]:
def design_matrix(x, degree=1):
    """Build the design matrix with a leading column of 1s.

    For degree=1: columns are [1, x] (intercept + slope).
    For degree=d: columns are [1, x, x^2, ..., x^d] (intercept + polynomial basis).
    """
    x = np.asarray(x, dtype=float)
    cols = [np.ones_like(x)] + [x ** k for k in range(1, degree + 1)]
    return np.column_stack(cols)


# Quick illustration on a small example
x_demo = np.array([0.5, 1.0, 1.5, 2.0])
print("Original x values:")
print(x_demo)
print()
print("Design matrix with intercept (degree=1):")
print(design_matrix(x_demo, degree=1))

Original x values:
[0.5 1.  1.5 2. ]

Design matrix with intercept (degree=1):
[[1.  0.5]
 [1.  1. ]
 [1.  1.5]
 [1.  2. ]]


Now we rewrite the risk and gradient to use the design matrix. The math is exactly the same — only the notation generalizes:

$$R(\theta) = \frac{1}{n} \| X\theta - y \|^2 \quad,\quad
  \nabla R(\theta) = \frac{2}{n} X^\top (X\theta - y)$$

In [9]:
def empirical_risk(X, y, theta):
    """Mean squared error using the design matrix."""
    return float(np.mean((X @ theta - y) ** 2))


def gradient(X, y, theta):
    """Vectorized gradient: (2/n) X^T (X theta - y)."""
    n = len(y)
    return (2 / n) * X.T @ (X @ theta - y)


def gradient_descent(X, y, alpha=0.05, n_iter=2000, theta0=None):
    """Batch GD. Returns final theta + full trajectory."""
    p = X.shape[1]
    theta = np.zeros(p) if theta0 is None else np.array(theta0, dtype=float)
    history = {"iter": [], "risk": [], "theta": []}
    for t in range(n_iter):
        theta = theta - alpha * gradient(X, y, theta)
        history["iter"].append(t)
        history["risk"].append(empirical_risk(X, y, theta))
        history["theta"].append(theta.copy())
    history["theta"] = np.array(history["theta"])
    return theta, history


# Build the design matrix and run GD again, this time with intercept.
X = design_matrix(x_arr, degree=1)
print(f"Design matrix shape: {X.shape}")
print(f"First 3 rows:")
print(X[:3])
print()
theta_gd, hist = gradient_descent(X, y_arr, alpha=0.05, n_iter=2000)
print(f"Final theta after 2000 iterations: {theta_gd}")
print(f"  theta_0 (intercept) = {theta_gd[0]:.2f}  (true value: 509)")
print(f"  theta_1 (slope)     = {theta_gd[1]:.2f}  (true value: 509)")
print(f"Final risk: {hist['risk'][-1]:.4f}")

Design matrix shape: (100, 2)
First 3 rows:
[[1.         1.5117461 ]
 [1.         0.50175656]
 [1.         1.41167557]]

Final theta after 2000 iterations: [510.43429647 507.86858468]
  theta_0 (intercept) = 510.43  (true value: 509)
  theta_1 (slope)     = 507.87  (true value: 509)
Final risk: 398.6252


In [10]:
fig = px.line(x=hist["iter"], y=hist["risk"], log_y=True,
              labels={"x": "iteration", "y": "empirical risk (log scale)"},
              title="GD convergence with intercept (alpha=0.05)")
fig.update_traces(line=dict(color=ARM_BLUE, width=2))
fig.update_layout(width=800, height=420)
fig

In [11]:
y_hat_grid = design_matrix(x_grid, degree=1) @ theta_gd

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_arr, y=y_arr, mode="markers",
                         marker=dict(color=ARM_BLUE, size=7, opacity=0.55),
                         name="data"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_grid, mode="lines",
                         line=dict(color=ARM_RED, width=3),
                         name=f"fit with intercept: y = {theta_gd[0]:.1f} + {theta_gd[1]:.1f} * x"))
fig.update_layout(title="Linear regression with intercept — recovers the true coefficients",
                  xaxis_title="x", yaxis_title="y",
                  width=800, height=450)
fig

### Side-by-side: with intercept vs without

Both fits on the same axes makes the difference unmissable. We extend the x-axis down to 0 so the no-intercept line visibly hits the origin while the with-intercept line crosses the y-axis at the right place.

In [12]:
y_hat_1p_grid = theta_1p * x_grid
y_hat_intercept_grid = design_matrix(x_grid, degree=1) @ theta_gd

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_arr, y=y_arr, mode="markers",
                         marker=dict(color=ARM_BLUE, size=7, opacity=0.55),
                         name="data"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_1p_grid, mode="lines",
                         line=dict(color=ARM_ORANGE, width=3, dash="dot"),
                         name=f"no intercept: y = {theta_1p:.0f} * x"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_intercept_grid, mode="lines",
                         line=dict(color=ARM_RED, width=3),
                         name=f"with intercept: y = {theta_gd[0]:.0f} + {theta_gd[1]:.0f} * x"))
# Mark the origin to make the "forced through (0,0)" point clear
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers",
                         marker=dict(color="black", size=10, symbol="x"),
                         name="(0, 0)"))
fig.update_layout(title="With intercept vs without — same data, two fits",
                  xaxis_title="x", yaxis_title="y",
                  width=800, height=450)
fig

The orange dotted line is forced through $(0, 0)$ and compensates with a steeper slope (~880); the red line is free to shift up to its true intercept (~509) and ends up with the correct slope (~509) too. From now on we always use the design matrix with a column of 1s.

## 4. Tune the learning rate

We sweep $\alpha$ on a log scale: too small → painfully slow; too large → divergence.

We plot **all** trajectories (including the divergent ones) below. The y-axis is log scale, so the divergent runs will sit at the top of the chart; use plotly's zoom (drag a rectangle, or scroll on the y-axis) to inspect the well-behaved $\alpha$ values.

In [13]:
alphas = [0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
histories = {}
for a in alphas:
    try:
        _, h = gradient_descent(X, y_arr, alpha=a, n_iter=300)
    except (FloatingPointError, OverflowError):
        h = None
    histories[a] = h

for a, h in histories.items():
    if h is None or not np.isfinite(h["risk"][-1]) or h["risk"][-1] > 1e10:
        final = h["risk"][-1] if h is not None else "crashed"
        print(f"  alpha={a:<6}: DIVERGED (final risk = {final})")
    else:
        print(f"  alpha={a:<6}: converged, final risk = {h['risk'][-1]:.4f}")

  alpha=0.001 : converged, final risk = 79447.2781
  alpha=0.01  : converged, final risk = 512.5048
  alpha=0.05  : converged, final risk = 398.6858
  alpha=0.1   : converged, final risk = 398.6252
  alpha=0.5   : DIVERGED (final risk = 6.935431617687433e+59)
  alpha=1.0   : DIVERGED (final risk = inf)


C:\Users\hayk_\AppData\Local\Programs\Python\Python310\lib\site-packages\numpy\_core\_methods.py:135: RuntimeWarning: overflow encountered in reduce
  ret = umr_sum(arr, axis, dtype, out, keepdims, where=where)
C:\Users\hayk_\AppData\Local\Temp\ipykernel_1880\1463090196.py:3: RuntimeWarning: overflow encountered in square
  return float(np.mean((X @ theta - y) ** 2))


In [14]:
# Plot ALL alpha sweeps — the divergent ones blow up the y-axis,
# but plotly is interactive so you can zoom in on the converging traces.
fig = go.Figure()
palette = [ARM_RED, ARM_BLUE, ARM_ORANGE, "purple", "gray", "black"]
for a, c in zip(alphas, palette):
    h = histories[a]
    if h is None:
        continue
    iters = np.array(h["iter"])
    risks = np.array(h["risk"])
    # Drop only NaN/inf (plotly can't plot those). Keep finite values
    # however large — user will zoom interactively.
    mask = np.isfinite(risks)
    fig.add_trace(go.Scatter(x=iters[mask], y=risks[mask], mode="lines",
                             line=dict(color=c, width=2), name=f"alpha={a}"))
fig.update_yaxes(type="log", title="empirical risk (log scale)")
fig.update_xaxes(title="iteration")
fig.update_layout(title="Risk vs iteration for different learning rates "
                        "(zoom into the bottom to see the converging traces)",
                  width=850, height=500)
fig

**Reading the plot.**

- Very small $\alpha$ (0.001) crawls down slowly — would need many more iterations.
- $\alpha \in \{0.05, 0.1\}$ converges smoothly and quickly to the bottom.
- $\alpha = 0.5$ keeps the risk roughly bounded but doesn't really minimize.
- $\alpha = 1.0$ blows up — risk shoots to $10^{200}$+ within a handful of iterations. The trace climbs almost vertically off the top of the chart.

## 5. A quick taste of SGD

Up to now we use **batch GD**: every step uses the gradient computed on *all* `n` samples. On huge datasets that's wasteful — we can update after looking at just a small batch (`batch_size << n`).

The price: each step is noisy because it's a sample of the true gradient. The benefit: many cheap steps per pass through the data, and the noise can help escape shallow saddle points.

In [15]:
def stochastic_gradient_descent(X, y, alpha=0.05, n_iter=2000, batch_size=1, seed=509):
    """Mini-batch SGD: pick `batch_size` random samples per step."""
    rng = np.random.default_rng(seed)
    p, n = X.shape[1], len(y)
    theta = np.zeros(p)
    history = {"iter": [], "risk": []}
    for t in range(n_iter):
        idx = rng.choice(n, size=batch_size, replace=False)
        Xb, yb = X[idx], y[idx]
        grad = (2 / len(yb)) * Xb.T @ (Xb @ theta - yb)
        theta = theta - alpha * grad
        history["iter"].append(t)
        history["risk"].append(empirical_risk(X, y, theta))  # measure on the FULL data
    return theta, history


theta_sgd, hist_sgd = stochastic_gradient_descent(X, y_arr, alpha=0.05, n_iter=2000, batch_size=1)
theta_mini, hist_mini = stochastic_gradient_descent(X, y_arr, alpha=0.05, n_iter=2000, batch_size=10)

print(f"Final theta from batch GD:        {theta_gd}")
print(f"Final theta from SGD (b=1):       {theta_sgd}")
print(f"Final theta from mini-batch (10): {theta_mini}")

Final theta from batch GD:        [510.43429647 507.86858468]
Final theta from SGD (b=1):       [507.35053629 513.86033242]
Final theta from mini-batch (10): [508.64651119 508.48080157]


In [16]:
fig = go.Figure()
fig.add_trace(go.Scatter(x=hist["iter"], y=hist["risk"], mode="lines",
                         line=dict(color=ARM_BLUE, width=2), name="batch GD (n=100)"))
fig.add_trace(go.Scatter(x=hist_sgd["iter"], y=hist_sgd["risk"], mode="lines",
                         line=dict(color=ARM_RED, width=1), name="SGD (batch=1)"))
fig.add_trace(go.Scatter(x=hist_mini["iter"], y=hist_mini["risk"], mode="lines",
                         line=dict(color=ARM_ORANGE, width=1.5), name="mini-batch (batch=10)"))
fig.update_yaxes(type="log", title="risk on full data (log scale)")
fig.update_xaxes(title="iteration")
fig.update_layout(title="Batch GD vs SGD vs mini-batch", width=850, height=450)
fig

SGD's risk curve is jagged — each step uses a single noisy estimate of the gradient — but it still settles near the minimum. Mini-batch (`batch=10`) smooths out the noise and is what most modern frameworks default to.

## 6. Normal equation — closed form

From section 3 we have $\nabla R(\theta) = \frac{2}{n} X^\top (X\theta - y)$. Set $\nabla R = 0$ and solve:

$$X^\top X \theta = X^\top y \quad \Rightarrow \quad
  \hat \theta = (X^\top X)^{-1} X^\top y$$

If $X$ has full column rank, the solution is unique and equals the global minimum of the risk.

In [17]:
def normal_equation(X, y):
    """Closed-form OLS via the normal equation.

    Uses np.linalg.solve when X.T @ X is well-conditioned. When the system
    is too ill-conditioned (e.g. high-degree polynomial features), falls
    back to np.linalg.lstsq, which uses an SVD that's more numerically
    stable.
    """
    try:
        return np.linalg.solve(X.T @ X, X.T @ y)
    except np.linalg.LinAlgError:
        return np.linalg.lstsq(X, y, rcond=None)[0]


theta_ne = normal_equation(X, y_arr)
print(f"theta from normal equation:  {theta_ne}")
print(f"theta from gradient descent: {theta_gd}")
print(f"difference:                  {theta_ne - theta_gd}")
print()
print(f"risk(theta_ne) = {empirical_risk(X, y_arr, theta_ne):.6f}")
print(f"risk(theta_gd) = {empirical_risk(X, y_arr, theta_gd):.6f}")

theta from normal equation:  [510.43429647 507.86858468]
theta from gradient descent: [510.43429647 507.86858468]
difference:                  [-5.68434189e-13  2.84217094e-13]

risk(theta_ne) = 398.625223
risk(theta_gd) = 398.625223


GD lands within rounding distance of the closed-form solution. The tiny difference comes from GD stopping at a finite number of iterations.

### Geometric sanity check

The normal equation $X^\top (X\hat\theta - y) = 0$ literally says **the residual vector is orthogonal to every column of $X$**. Let's verify.

In [18]:
residuals_ne = y_arr - X @ theta_ne
ortho_check = X.T @ residuals_ne
print("X.T @ residuals (should be approximately zero):")
print(ortho_check)
print(f"\nmax |entry| = {np.abs(ortho_check).max():.2e}")

X.T @ residuals (should be approximately zero):
[ 5.34328137e-11 -9.59232693e-12]

max |entry| = 5.34e-11


Both entries are numerically zero (within floating-point precision). Geometrically: the column space of $X$ is a 2-D plane sitting in $\mathbb R^n$, and $\hat y = X\hat\theta$ is the orthogonal projection of $y$ onto that plane. The residual is what's left over after the projection — and it's perpendicular to the plane by construction.

## 7. Use the model — predict on new x

In [19]:
y_hat_grid = design_matrix(x_grid, degree=1) @ theta_ne

fig = go.Figure()
fig.add_trace(go.Scatter(x=x_arr, y=y_arr, mode="markers",
                         marker=dict(color=ARM_BLUE, size=7, opacity=0.55),
                         name="data"))
fig.add_trace(go.Scatter(x=x_arr, y=X @ theta_ne, mode="markers",
                         marker=dict(color=ARM_ORANGE, size=6, symbol="x"),
                         name="predictions on training x"))
fig.add_trace(go.Scatter(x=x_grid, y=y_hat_grid, mode="lines",
                         line=dict(color=ARM_RED, width=3),
                         name="fitted line"))
fig.update_layout(title="Linear regression fit",
                  xaxis_title="x", yaxis_title="y",
                  width=800, height=450)
fig

theta_0_hat, theta_1_hat = theta_ne
print(f"Final formula:  y_hat = {theta_0_hat:.2f} + {theta_1_hat:.2f} * x")

Final formula:  y_hat = 510.43 + 507.87 * x


A trained model is supposed to *do something* — predict for inputs it hasn't seen. Let's plug in a few new $x$ values:

In [20]:
new_x = np.array([0.25, 0.75, 1.30, 1.85])
new_X = design_matrix(new_x, degree=1)
new_y_hat = new_X @ theta_ne

print("Predictions on new x values:")
for x_val, y_val in zip(new_x, new_y_hat):
    print(f"  x = {x_val:.2f}  ->  y_hat = {y_val:7.2f}")

Predictions on new x values:
  x = 0.25  ->  y_hat =  637.40
  x = 0.75  ->  y_hat =  891.34
  x = 1.30  ->  y_hat = 1170.66
  x = 1.85  ->  y_hat = 1449.99


## 8. Polynomial regression — and what breaks numerically

The exact same model can fit non-linear curves if we just give it different features. To fit a degree-$d$ polynomial we add $x^2, x^3, \ldots, x^d$ as new columns of the design matrix. Let's look at what that matrix actually looks like.

In [21]:
x_demo = np.array([0.5, 1.0, 1.5, 2.0])

print("Original x values:")
print(x_demo)
print()
for d in [1, 2, 3, 5]:
    Xd_demo = design_matrix(x_demo, d)
    print(f"design_matrix(x, degree={d})   shape = {Xd_demo.shape}")
    print(f"  columns are: 1, x" + ("".join(f", x^{k}" for k in range(2, d + 1))))
    print(Xd_demo)
    print()

Original x values:
[0.5 1.  1.5 2. ]

design_matrix(x, degree=1)   shape = (4, 2)
  columns are: 1, x
[[1.  0.5]
 [1.  1. ]
 [1.  1.5]
 [1.  2. ]]

design_matrix(x, degree=2)   shape = (4, 3)
  columns are: 1, x, x^2
[[1.   0.5  0.25]
 [1.   1.   1.  ]
 [1.   1.5  2.25]
 [1.   2.   4.  ]]

design_matrix(x, degree=3)   shape = (4, 4)
  columns are: 1, x, x^2, x^3
[[1.    0.5   0.25  0.125]
 [1.    1.    1.    1.   ]
 [1.    1.5   2.25  3.375]
 [1.    2.    4.    8.   ]]

design_matrix(x, degree=5)   shape = (4, 6)
  columns are: 1, x, x^2, x^3, x^4, x^5
[[1.00000e+00 5.00000e-01 2.50000e-01 1.25000e-01 6.25000e-02 3.12500e-02]
 [1.00000e+00 1.00000e+00 1.00000e+00 1.00000e+00 1.00000e+00 1.00000e+00]
 [1.00000e+00 1.50000e+00 2.25000e+00 3.37500e+00 5.06250e+00 7.59375e+00]
 [1.00000e+00 2.00000e+00 4.00000e+00 8.00000e+00 1.60000e+01 3.20000e+01]]



So `degree=d` just stacks more columns. The model itself doesn't change — we still solve the same normal equation, only the matrix is wider.

But there's a hidden cost. As the degree grows, $X^\top X$ becomes increasingly **ill-conditioned**: its rows / columns get nearly linearly dependent (because $x, x^2, x^3$ are correlated for similar $x$ values), and `np.linalg.solve` starts losing significant digits. We can measure this with the **condition number** — the ratio between the largest and smallest singular values of $X^\top X$. Anything above $\sim 10^{10}$ means we're in trouble.

**For this demonstration we fit on a random subset of 30 points** — with the full 100 points there are so many constraints that even a degree-10 polynomial can't wiggle enough to look dramatic. Fewer points = more freedom for the polynomial to bend between them. The wiggling you'll see is exactly what an unregularized high-degree model does on small datasets.

In [22]:
# Random subset of 30 points - polynomial fits will have more freedom to wiggle
rng_sub = np.random.default_rng(SEED)
subset_idx = rng_sub.choice(len(x_arr), size=30, replace=False)
x_sub = x_arr[subset_idx]
y_sub = y_arr[subset_idx]
print(f"Subset shape: {x_sub.shape}")

degrees = [1, 3, 10, 50]
fits = {}
risks_by_d = {}
conds_by_d = {}
for d in degrees:
    Xd = design_matrix(x_sub, degree=d)
    theta_d = normal_equation(Xd, y_sub)
    fits[d] = theta_d
    risks_by_d[d] = empirical_risk(Xd, y_sub, theta_d)
    conds_by_d[d] = np.linalg.cond(Xd.T @ Xd)

Subset shape: (30,)


In [23]:
# Dense grid so the curves look smooth even when wiggling fast
x_grid_dense = np.linspace(x_arr.min(), x_arr.max(), 1000)

# Zoom window — center of the data range, where the wiggling is most visible.
ZOOM_X = (0.5, 1.5)
zoom_mask = (x_sub >= ZOOM_X[0]) & (x_sub <= ZOOM_X[1])
y_zoom = y_sub[zoom_mask] if zoom_mask.any() else y_sub
y_pad = (y_zoom.max() - y_zoom.min()) * 0.25
ZOOM_Y = (y_zoom.min() - y_pad, y_zoom.max() + y_pad)

fig = make_subplots(rows=2, cols=2,
                    subplot_titles=[f"degree={d}, risk={risks_by_d[d]:.1f}, cond(X^T X)={conds_by_d[d]:.0e}"
                                    for d in degrees],
                    shared_xaxes=True, shared_yaxes=True,
                    horizontal_spacing=0.07, vertical_spacing=0.13)
for i, d in enumerate(degrees):
    row, col = i // 2 + 1, i % 2 + 1
    Xd_grid = design_matrix(x_grid_dense, degree=d)
    y_hat = Xd_grid @ fits[d]
    # All 100 original data points (lightly), subset highlighted
    fig.add_trace(go.Scatter(x=x_arr, y=y_arr, mode="markers",
                             marker=dict(color="lightgray", size=4, opacity=0.5),
                             showlegend=False, hoverinfo="skip"), row=row, col=col)
    fig.add_trace(go.Scatter(x=x_sub, y=y_sub, mode="markers",
                             marker=dict(color=ARM_BLUE, size=6, opacity=0.85),
                             showlegend=False, name="fit points"), row=row, col=col)
    fig.add_trace(go.Scatter(x=x_grid_dense, y=y_hat, mode="lines",
                             line=dict(color=ARM_RED, width=2),
                             showlegend=False), row=row, col=col)
fig.update_xaxes(range=[ZOOM_X[0], ZOOM_X[1]])
fig.update_yaxes(range=[ZOOM_Y[0], ZOOM_Y[1]])
fig.update_layout(title=f"Polynomial regression on a 30-point subset "
                        f"(blue = fit points, gray = other data) — zoomed to x in [{ZOOM_X[0]}, {ZOOM_X[1]}]",
                  width=900, height=620)
fig

In [24]:
print("degree | train risk | cond(X^T X)")
print("-" * 40)
for d in degrees:
    print(f"  {d:2d}   |  {risks_by_d[d]:8.2f}  | {conds_by_d[d]:.2e}")

degree | train risk | cond(X^T X)
----------------------------------------
   1   |    440.45  | 1.62e+01
   3   |    433.60  | 1.21e+04
  10   |    225.94  | 1.18e+16
  50   |    160.22  | 7.19e+37


**Reading the plot.** With 30 points the polynomials have visible room to wiggle:

- $d = 1$: clean straight line, lowest condition number.
- $d = 3$: still smooth, captures any subtle curvature.
- $d = 10$: starts visibly bending between points. The line shouldn't be doing that — there are only ~10 fit points in this zoom and the polynomial has 11 coefficients, so it has enough capacity to weave around them.
- $d = 50$: $n = 30 < p = 51$, the system is underdetermined. `lstsq` returns the minimum-norm solution; the curve passes through every fit point exactly and oscillates wildly between them. Cond$(X^\top X) > 10^{30}$ — many of the digits in `theta_d` are pure floating-point noise.

Later in the course we'll meet **regularization** (Ridge, Lasso): it adds $\lambda I$ to $X^\top X$ before inverting, which stabilizes the condition number AND keeps the coefficients reasonable.

## Bonus — residual histogram

In [25]:
residuals = y_arr - X @ theta_ne
print(f"residuals: mean={residuals.mean():.3f}, std={residuals.std():.3f}, "
      f"min={residuals.min():.2f}, max={residuals.max():.2f}")

fig = px.histogram(x=residuals, nbins=25,
                   labels={"x": "residual = y - y_hat", "y": "count"},
                   title="Residuals of the linear fit")
fig.update_traces(marker=dict(color=ARM_BLUE, opacity=0.8, line=dict(color="white", width=1)))
fig.add_vline(x=0, line=dict(color=ARM_RED, width=2), annotation_text="zero")
fig.update_layout(width=800, height=400)
fig

residuals: mean=0.000, std=19.966, min=-59.77, max=57.53


Residuals look roughly symmetric and centered near zero, with a standard deviation close to 20 (the noise level we put in when generating the data). The model has extracted the systematic signal and left only the random noise.

## Bonus — risk surface and GD trajectory

We plot $R(\theta_0, \theta_1)$ as a contour. The risk grows fast as we move away from the optimum (it's a quadratic in $\theta$), so we use $\log_{10}(R)$ as the color so the contours are visible across the full dynamic range. We overlay the path GD takes from $\theta = 0$ down to the minimum.

In [26]:
# Build a grid around the optimum
t0_opt, t1_opt = theta_ne
t0_range = np.linspace(t0_opt - 300, t0_opt + 300, 100)
t1_range = np.linspace(t1_opt - 400, t1_opt + 400, 100)
T0, T1 = np.meshgrid(t0_range, t1_range)

R = np.empty_like(T0)
for i in range(T0.shape[0]):
    for j in range(T0.shape[1]):
        theta = np.array([T0[i, j], T1[i, j]])
        R[i, j] = np.mean((X @ theta - y_arr) ** 2)

# Trajectory from the converging alpha
traj = histories[0.05]["theta"]

In [27]:
log_R = np.log10(R)

fig = go.Figure()
# Heatmap coloring fills the whole grid (no white corners). Contour lines on top.
fig.add_trace(go.Contour(
    x=t0_range, y=t1_range, z=log_R,
    colorscale="Viridis",
    contours=dict(
        coloring="heatmap",   # fill the entire grid, not just bands between lines
        start=log_R.min(), end=log_R.max(),
        size=(log_R.max() - log_R.min()) / 20,
        showlines=True,
        showlabels=False,
    ),
    line=dict(width=0.5, color="rgba(255,255,255,0.4)"),
    colorbar=dict(title="log10(risk)", thickness=15, x=1.02),
))
# GD trajectory: thin red line so it doesn't dominate the optimum marker
fig.add_trace(go.Scatter(x=traj[:, 0], y=traj[:, 1], mode="lines",
                         line=dict(color=ARM_RED, width=2.5),
                         name="GD trajectory (alpha=0.05)"))
# Start point: square marker with white fill, easy to find
fig.add_trace(go.Scatter(x=[0], y=[0], mode="markers",
                         marker=dict(color="white", size=14, symbol="square",
                                     line=dict(color="black", width=2)),
                         name="start theta=(0, 0)"))
# OLS optimum: big gold star on top of everything
fig.add_trace(go.Scatter(x=[t0_opt], y=[t1_opt], mode="markers",
                         marker=dict(color=ARM_ORANGE, size=22, symbol="star",
                                     line=dict(color="black", width=1.5)),
                         name="OLS optimum"))
fig.update_layout(title="Risk surface (log10 scale) + gradient-descent trajectory",
                  xaxis_title="theta_0 (intercept)",
                  yaxis_title="theta_1 (slope)",
                  width=900, height=620,
                  legend=dict(
                      orientation="h",
                      yanchor="bottom", y=-0.18,
                      xanchor="center", x=0.5,
                      bgcolor="rgba(255,255,255,0.7)",
                  ),
                  margin=dict(l=70, r=110, t=70, b=80))
fig

GD walks downhill from $(0, 0)$ to the OLS minimum, with each step approximately perpendicular to the contour lines (the negative gradient direction). The starting position (white square in the corner) is far from the optimum (gold star near the center) — about 700+ units away in $\theta$-space.

## Bonus — vectorization sanity check

Every function above is vectorized: predictions are `X @ theta`, the gradient is `(2/n) * X.T @ (X @ theta - y)`, no Python-level loops over samples. Compared to a sample-by-sample loop the speed difference is dramatic — let's measure it.

In [28]:
import time

def gradient_loop(X, y, theta):
    """Same gradient computed via an explicit per-sample loop."""
    n = len(y)
    g = np.zeros_like(theta)
    for i in range(n):
        diff = X[i] @ theta - y[i]
        g += diff * X[i]
    return (2 / n) * g


# Replicate the data 1000x so the timing is non-trivial
X_big = np.tile(X, (1000, 1))
y_big = np.tile(y_arr, 1000)
print(f"timing on {X_big.shape[0]:,} rows")

t0 = time.time(); _ = gradient(X_big, y_big, theta_ne); t_vec = time.time() - t0
t0 = time.time(); _ = gradient_loop(X_big, y_big, theta_ne); t_loop = time.time() - t0
print(f"  vectorized: {t_vec*1000:7.2f} ms")
print(f"  loop:       {t_loop*1000:7.2f} ms")
print(f"  speedup:    {t_loop/t_vec:7.1f}x")

timing on 100,000 rows


  vectorized:    1.03 ms
  loop:        309.78 ms
  speedup:      300.4x


The vectorized version is typically two to three orders of magnitude faster on this size of data — BLAS / SIMD doing the work in C, no Python interpreter overhead per sample.

## Recap

- Generated our own data with known true coefficients ($\theta_0 = 509$, $\theta_1 = 509$) so we could check the model recovers them.
- Started with the simplest model $\hat y = \theta x$ and derived gradient descent for a single parameter — saw it fail when the data doesn't pass through the origin.
- Introduced the **design-matrix trick** (prepend a column of 1s) to add an intercept without writing special-case code. The fit recovers the true coefficients almost exactly, and a side-by-side plot made the comparison vivid.
- Tuned the learning rate and saw what happens at the boundaries (too small = slow; too large = explode). Plotted *all* trajectories so the divergence is visible.
- Demonstrated **SGD and mini-batch GD** as the practical big-data variants of vanilla GD.
- Solved the same problem in closed form via the **normal equation**, with a robust `lstsq` fallback for high-degree systems, and verified its geometric content: residuals are orthogonal to the column space of $X$.
- Used the trained model to predict on new $x$ values — the actual job a regression model exists to do.
- Extended to polynomial regression at degrees $\{1, 3, 10, 50\}$, on a 30-point random subset so the wiggling actually shows up — and measured the **condition number** blowing up as a numerical warning that motivates regularization.
- Visualized GD as a walk down the risk surface.
- Demonstrated that vectorization is what makes this practical at scale.